# 🦷 PARA ALUMNOS — Entrena tu propia IA de Diagnóstico
### Seminario: IA y Seguridad de Datos en Odontología
**Ponente:** Marco Antonio Robles Huamán — Keluma SAC

---
| | Paso | Descripción | Tiempo |
|--|------|-------------|--------|
| ⚙️ | Configuración | Elige tu rol y carpeta | 30 seg |
| 🔄 | Limpiar | Opcional — resetear datos | 10 seg |
| **▶** | **PASO 1** | **Setup + Convertir DICOMs → PNGs + Generar Excel** | **2-5 min** |
| **✋** | **PASO 2** | **Tú llenas el Excel con diagnósticos** | **15-30 min** |
| **▶** | **PASO 3** | **Entrenar modelo IA con tus imágenes** | **5-10 min** |
| 🔍 | APP | Diagnosticar imagen nueva | inmediato |

---
### 📁 Antes de empezar:
Crea esta carpeta en **tu Google Drive**:
```
cursos/curso odontologia/dataset_dental/dicoms/
```
Y pon tus archivos `.dcm` dentro. Si no tienes DICOMs propios, el profesor te dará un set de prueba.


In [ ]:
from PIL import Image
#@title ⚙️ CONFIGURACIÓN — Ejecutar primero
import subprocess, sys

ROL = "alumno"  #@param ["alumno", "profesor"]
#@markdown 👤 **alumno** = usa tu carpeta `dataset_dental` | **profesor** = usa `dataset_profesor`

CARPETA_DATASET = "cursos/curso odontologia/dataset_dental"  #@param {type:"string"}
#@markdown 📂 Cambia esto solo si tu carpeta tiene otro nombre

RESOLUCION = 128  #@param [64, 128, 256]
#@markdown 🖼️ Resolución de imágenes (128 recomendado)

EPOCHS = 15  #@param {type:"slider", min:5, max:30, step:5}
#@markdown 🔄 Épocas de entrenamiento

# Ajuste automático por rol
if ROL == "profesor":
    CARPETA_DATASET = "cursos/curso odontologia/dataset_profesor"
    EPOCHS = max(EPOCHS, 20)
    print("   ⚠️ ROL PROFESOR: usando dataset_profesor")

print("📦 Instalando librerías...")
subprocess.run([sys.executable,"-m","pip","install",
    "pydicom","pillow","openpyxl","tensorflow",
    "scikit-learn","matplotlib","--quiet"],check=True)
print("✅ Librerías listas")

from google.colab import drive
print("\n📁 Conectando Google Drive...")
drive.mount("/content/drive")

import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

BASE     = "/content/drive/MyDrive"
RUTA_DS  = f"{BASE}/{CARPETA_DATASET}"
RUTA_DCM = f"{RUTA_DS}/dicoms"
RUTA_PNG = f"{RUTA_DS}/pngs"
RUTA_MOD = f"{RUTA_DS}/modelo"
RUTA_RES = f"{RUTA_DS}/resultados"
RUTA_XLS = f"{RUTA_DS}/diagnosticos.xlsx"

for r in [RUTA_PNG, RUTA_MOD, RUTA_RES]:
    os.makedirs(r, exist_ok=True)

if not os.path.exists(RUTA_DCM):
    print(f"\n❌ No se encontró la carpeta de DICOMs:")
    print(f"   {RUTA_DCM}")
    print(f"   Créala y sube tus archivos .dcm antes de continuar.")
else:
    n_dcm = len([f for f in os.listdir(RUTA_DCM) if f.lower().endswith(".dcm")])
    print(f"\n✅ Configuración lista")
    print(f"   ROL      : {ROL}")
    print(f"   Dataset  : {RUTA_DS}")
    print(f"   DICOMs   : {n_dcm} archivos encontrados")
    print(f"   Épocas   : {EPOCHS}")


In [ ]:
#@title 🔄 LIMPIAR — Opcional: resetear para cargar nuevos datos
QUE_LIMPIAR = "solo_modelo_y_resultados"  #@param ["solo_modelo_y_resultados", "pngs_modelo_y_resultados", "todo_excepto_dicoms"]
#@markdown ⚠️ Usa **todo_excepto_dicoms** si cambiaste tus imágenes o quieres empezar de cero

import shutil
if QUE_LIMPIAR in ["solo_modelo_y_resultados","pngs_modelo_y_resultados","todo_excepto_dicoms"]:
    shutil.rmtree(RUTA_MOD,ignore_errors=True); os.makedirs(RUTA_MOD,exist_ok=True)
    shutil.rmtree(RUTA_RES,ignore_errors=True); os.makedirs(RUTA_RES,exist_ok=True)
if QUE_LIMPIAR in ["pngs_modelo_y_resultados","todo_excepto_dicoms"]:
    shutil.rmtree(RUTA_PNG,ignore_errors=True); os.makedirs(RUTA_PNG,exist_ok=True)
if QUE_LIMPIAR == "todo_excepto_dicoms":
    if os.path.exists(RUTA_XLS): os.remove(RUTA_XLS)
    print("✅ Todo limpio — listo para empezar de cero")
else:
    print(f"✅ Limpieza: {QUE_LIMPIAR}")


---
# ▶ PASO 1 — Preparar mis datos
> Convierte tus DICOMs a PNG y genera el Excel para etiquetar.
> **No necesitas hacer nada más** — solo espera a que termine.

In [ ]:
#@title ▶ PASO 1 — Convertir DICOMs → PNGs + Generar Excel
import pydicom, openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from PIL import Image

print("━"*55)
print("  ▶  PASO 1 — PREPARAR MIS DATOS")
print("━"*55)

# --- Convertir DICOMs ---
archivos = sorted([f for f in os.listdir(RUTA_DCM) if f.lower().endswith(".dcm")])
print(f"\n   📂 {len(archivos)} DICOMs encontrados")
print(f"   🔄 Convirtiendo a PNG {RESOLUCION}×{RESOLUCION}px...")

def dicom_a_png(ruta_dcm, ruta_png, res):
    dcm = pydicom.dcmread(ruta_dcm)
    px  = dcm.pixel_array
    if px.ndim == 3: px = px[:,:,0]
    mn,mx = px.min(),px.max()
    if mx>mn: px=((px-mn)/(mx-mn)*255).astype(np.uint8)
    else:      px=px.astype(np.uint8)
    px = np.array(Image.fromarray(px).resize((res,res)))
    Image.fromarray(px,"L").save(ruta_png)
    return px

conv, err = [], []
for i,f in enumerate(archivos):
    png = f.replace(".dcm",".png").replace(".DCM",".png")
    try:
        dicom_a_png(f"{RUTA_DCM}/{f}", f"{RUTA_PNG}/{png}", RESOLUCION)
        conv.append(png)
    except Exception as e: err.append((f,str(e)))
    if (i+1)%50==0: print(f"   ✅ {i+1}/{len(archivos)}...")

print(f"\n   ✅ {len(conv)} PNGs generados en tu Drive")
if err: print(f"   ⚠️ {len(err)} errores")

# --- Generar Excel en Drive ---
print(f"\n   📋 Generando Excel de diagnósticos...")
wb = openpyxl.Workbook(); ws = wb.active; ws.title="diagnosticos"

YEL = PatternFill("solid",fgColor="FFF2CC")
GRY = PatternFill("solid",fgColor="D6DCE4")
ALT = PatternFill("solid",fgColor="FFFDE7")
brd = lambda: Border(left=Side(style="thin",color="AAAAAA"),right=Side(style="thin",color="AAAAAA"),
                     top=Side(style="thin",color="AAAAAA"),bottom=Side(style="thin",color="AAAAAA"))

COLS=[("archivo",28,GRY,"0D1B40"),("diagnostico",20,YEL,"7B3F00"),
      ("notas",35,ALT,"7B3F00"),("revisor",18,YEL,"7B3F00")]
ws.row_dimensions[1].height=20
for ci,(h,w,bg,fc) in enumerate(COLS,1):
    c=ws.cell(row=1,column=ci,value=h)
    c.fill=bg; c.font=Font(bold=True,size=11,color=fc)
    c.alignment=Alignment(horizontal="center",vertical="center"); c.border=brd()
    ws.column_dimensions[get_column_letter(ci)].width=w

for i,png in enumerate(conv):
    row=i+2; bg=YEL if i%2==0 else ALT
    ws.cell(row=row,column=1,value=png).fill=PatternFill("solid",fgColor="EBF5FB")
    ws.cell(row=row,column=1).font=Font(size=9,bold=True); ws.cell(row=row,column=1).border=brd()
    for ci in [2,3,4]:
        c=ws.cell(row=row,column=ci,value=""); c.fill=bg; c.font=Font(size=9); c.border=brd()
    ws.row_dimensions[row].height=13

ws.freeze_panes="A2"

# Hoja de instrucciones
wi=wb.create_sheet("INSTRUCCIONES")
instrucciones=[
    ("DIAGNÓSTICOS VÁLIDOS",""),
    ("caries","Lesión cariosa visible"),
    ("sano","Sin patología"),
    ("periapical","Lesión en ápice radicular"),
    ("reabsorcion","Reabsorción radicular"),
    ("fractura","Fractura dental"),
    ("otro","Otro hallazgo — especifica en notas"),
    ("",""),
    ("IMPORTANTE","Escríbelos exactamente: minúsculas, sin tildes, sin espacios extra"),
]
for i,(k,v) in enumerate(instrucciones,1):
    wi.cell(row=i,column=1,value=k).font=Font(bold=True if i==1 or not v else False,size=11)
    wi.cell(row=i,column=2,value=v).font=Font(size=10)
wi.column_dimensions["A"].width=18; wi.column_dimensions["B"].width=45

wb.save(RUTA_XLS)
print(f"   ✅ Excel guardado en Drive: diagnosticos.xlsx")

# Galería
muestra=conv[:15]; cols=5; rows=(len(muestra)+cols-1)//cols
fig,axes=plt.subplots(rows,cols,figsize=(cols*2.5,rows*2.5))
axes=np.array(axes).flatten()
fig.suptitle(f"Mis Radiografías — {len(conv)} imágenes listas",fontsize=13,fontweight="bold")
for j,nombre in enumerate(muestra):
    img=np.array(Image.open(f"{RUTA_PNG}/{nombre}"))
    axes[j].imshow(img,cmap="gray"); axes[j].set_title(nombre[:15],fontsize=7); axes[j].axis("off")
for j in range(len(muestra),len(axes)): axes[j].axis("off")
plt.tight_layout(); plt.show()

print("\n" + "━"*55)
print(f"  ✅  PASO 1 LISTO")
print(f"      {len(conv)} PNGs en Drive")
print(f"      Excel generado: diagnosticos.xlsx")
print("━"*55)
print()
print("  ┌─────────────────────────────────────────┐")
print("  │  ✋  AHORA VE A TU GOOGLE DRIVE         │")
print("  │                                         │")
print(f"  │  Carpeta: {CARPETA_DATASET[:35]}")
print("  │  Archivo: diagnosticos.xlsx             │")
print("  │                                         │")
print("  │  1. Ábrelo en Google Sheets o Excel     │")
print("  │  2. Revisa cada imagen en la carpeta    │")
print("  │     pngs/                               │")
print("  │  3. Escribe el diagnóstico en la        │")
print("  │     columna AMARILLA                    │")
print("  │  4. Guarda el archivo                   │")
print("  │  5. Regresa aquí y ejecuta el PASO 3   │")
print("  └─────────────────────────────────────────┘")


---
# ✋ PASO 2 — Llenar el Excel con mis diagnósticos

> **Este paso lo haces TÚ** — no hay código que ejecutar aquí.

### Instrucciones:
1. Abre tu **Google Drive**
2. Ve a la carpeta: `cursos/curso odontologia/dataset_dental/`
3. Abre el archivo `diagnosticos.xlsx` en Google Sheets o Excel
4. Para cada archivo, revisa la imagen en la carpeta `pngs/` y escribe el diagnóstico:

| Valor | Descripción |
|-------|-------------|
| `caries` | Lesión cariosa visible |
| `sano` | Sin patología |
| `periapical` | Lesión en ápice radicular |
| `reabsorcion` | Reabsorción radicular |
| `fractura` | Fractura dental |
| `otro` | Otro hallazgo |

> ⚠️ Escríbelos **exactamente así**: minúsculas, sin tildes, sin espacios extra.

### Cuando termines de llenar el Excel → **ejecuta el PASO 3**


In [ ]:
#@title ▶ PASO 3 — Entrenar mi modelo IA
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

print("━"*55)
print("  ▶  PASO 3 — ENTRENAR MI MODELO IA")
print("━"*55)

if not os.path.exists(RUTA_XLS):
    raise FileNotFoundError("No se encontró diagnosticos.xlsx. Ejecuta el PASO 1 primero.")

df = pd.read_excel(RUTA_XLS)
df.columns = [c.lower().strip() for c in df.columns]
VALIDOS = ["caries","sano","periapical","reabsorcion","fractura","otro"]
df_ok   = df[df["diagnostico"].isin(VALIDOS)].copy()

if len(df_ok) == 0:
    raise ValueError("El Excel no tiene diagnósticos válidos. Completa el PASO 2 primero.")

n_vacias = (df["diagnostico"].isna() | ~df["diagnostico"].isin(VALIDOS)).sum()
print(f"\n   📊 Resumen del Excel:")
print(f"      Total filas    : {len(df)}")
print(f"      Con diagnóstico: {len(df_ok)}")
if n_vacias: print(f"      Sin llenar     : {n_vacias} (se omiten)")

categorias = sorted(df_ok["diagnostico"].unique().tolist())
n_clases   = len(categorias)
cat2idx    = {c:i for i,c in enumerate(categorias)}

print(f"\n   Clases ({n_clases}):")
for c in categorias:
    n=(df_ok["diagnostico"]==c).sum()
    print(f"      {'█'*min(n,20)}  {c}: {n}")

# Cargar imágenes
print(f"\n   🖼️  Cargando imágenes...")
X,y,faltantes=[],[],[]
for _,row in df_ok.iterrows():
    fname=str(row["archivo"]).replace(".dcm",".png").replace(".DCM",".png")
    ruta=f"{RUTA_PNG}/{fname}"
    if not os.path.exists(ruta): faltantes.append(fname); continue
    img=np.array(Image.open(ruta).convert("L").resize((RESOLUCION,RESOLUCION)))
    rgb=np.stack([img,img,img],axis=-1).astype(np.float32)/255.0
    X.append(rgb); y.append(cat2idx[row["diagnostico"]])

X=np.array(X); y=np.array(y)
if faltantes: print(f"   ⚠️ {len(faltantes)} imágenes no encontradas (¿ejecutaste el PASO 1?)")
print(f"   ✅ {len(X)} imágenes cargadas")

if len(X)<10:
    raise ValueError("Necesitas al menos 10 imágenes etiquetadas para entrenar.")

X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,random_state=42,
                      stratify=y if len(X)>n_clases*2 else None)
print(f"   ✅ Train: {len(X_tr)} · Test: {len(X_te)}")

# Modelo MobileNetV2
print("\n   🏗️  Construyendo modelo...")
base=keras.applications.MobileNetV2(input_shape=(RESOLUCION,RESOLUCION,3),
     include_top=False,weights="imagenet")
base.trainable=False
model=keras.Sequential([base,layers.GlobalAveragePooling2D(),
      layers.Dense(128,activation="relu"),layers.Dropout(0.3),
      layers.Dense(n_clases,activation="softmax")])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
print(f"   ✅ Parámetros: {model.count_params():,}")

cb=[keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True,monitor="val_accuracy")]
print(f"\n   🔄 Entrenando {EPOCHS} épocas...")
history=model.fit(X_tr,y_tr,epochs=EPOCHS,batch_size=8,
                  validation_data=(X_te,y_te),callbacks=cb,verbose=1)

_,acc=model.evaluate(X_te,y_te,verbose=0)

# Guardar modelo
ruta_keras=f"{RUTA_MOD}/mi_modelo.keras"
model.save(ruta_keras)
with open(f"{RUTA_MOD}/categorias.json","w") as f:
    json.dump({"categorias":categorias,"resolucion":RESOLUCION},f)

# Curvas
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4))
for ax,m,t in [(a1,"accuracy","Accuracy"),(a2,"loss","Loss")]:
    ax.plot(history.history[m],"o-",color="#1A73E8",label="Train")
    ax.plot(history.history[f"val_{m}"],"s--",color="#C00000",label="Validación")
    ax.set_title(f"{t}",fontweight="bold"); ax.legend(); ax.grid(True,alpha=0.3)
fig.suptitle(f"Mi Modelo IA — Accuracy: {acc*100:.1f}%",fontsize=14,fontweight="bold",color="#0D1B40")
plt.tight_layout()
fig.savefig(f"{RUTA_RES}/curvas_entrenamiento.png",dpi=150,bbox_inches="tight")
plt.show()

print("\n"+"━"*55)
print(f"  ✅  PASO 3 LISTO — Tu modelo está entrenado")
print(f"      Accuracy : {acc*100:.1f}%")
print(f"      Keras    : {ruta_keras}")
print("  → Ejecuta la APP para diagnosticar imágenes nuevas")
print("━"*55)


---
# 🔍 APP — Diagnosticar Imagen Nueva
> Ejecuta la celda de **PASO 1** para ver los archivos disponibles.  
> Luego escribe el nombre del archivo en **PASO 2** y ejecuta para obtener el diagnóstico.

### Acepta:
- **DICOM (.dcm)** — muestra metadatos del paciente antes de anonimizar
- **PNG (.png)** — usa directamente (sin metadatos de paciente)
- **JPG (.jpg)** — usa directamente

> Puedes ejecutar el diagnóstico cuantas veces quieras con diferentes archivos.

In [ ]:
#@title 🔍 APP — Diagnosticar Imagen Nueva (DICOM o PNG)
import tensorflow as tf, datetime
from tensorflow import keras

# ── Listar archivos disponibles ──────────────────────
dcms = sorted([f for f in os.listdir(RUTA_DCM) if f.lower().endswith(".dcm")]) if os.path.exists(RUTA_DCM) else []
pngs = sorted([f for f in os.listdir(RUTA_PNG) if f.lower().endswith(".png")]) if os.path.exists(RUTA_PNG) else []

print("━"*55)
print("  🔍  APP — DIAGNÓSTICO DE IMAGEN NUEVA")
print("━"*55)
print()
if dcms:
    print(f"  📂 DICOMs disponibles en dicoms/ ({len(dcms)} archivos):")
    for f in dcms[:10]: print(f"     {f}")
    if len(dcms)>10: print(f"     ... y {len(dcms)-10} más")
    print()
if pngs:
    print(f"  🖼️  PNGs disponibles en pngs/ ({len(pngs)} archivos):")
    for f in pngs[:10]: print(f"     {f}")
    if len(pngs)>10: print(f"     ... y {len(pngs)-10} más")
    print()
print("  → Escribe el nombre del archivo abajo y ejecuta de nuevo")
print("━"*55)


In [ ]:
#@title 🔍 APP — Selecciona y diagnostica
ARCHIVO   = "rx_0001_normal.dcm"  #@param {type:"string"}
#@markdown 📂 Nombre del archivo (de dicoms/ o pngs/) o ruta completa en Drive

print("━"*55)
print("  🔍  DIAGNOSTICANDO:", ARCHIVO)
print("━"*55)

# Cargar modelo
ruta_keras = f"{RUTA_MOD}/mi_modelo.keras"
ruta_meta  = f"{RUTA_MOD}/categorias.json"
if not os.path.exists(ruta_keras):
    raise FileNotFoundError("No hay modelo. Ejecuta el PASO 2 (profesor) o PASO 3 (alumno) primero.")

model_app = keras.models.load_model(ruta_keras)
with open(ruta_meta) as f: meta = json.load(f)
cats = meta["categorias"]; res = meta["resolucion"]
print(f"   ✅ Modelo cargado — clases: {cats}")

# Resolver ruta del archivo
import pydicom
if os.path.isabs(ARCHIVO) or ARCHIVO.startswith("/content"):
    ruta_archivo = ARCHIVO
elif os.path.exists(f"{RUTA_DCM}/{ARCHIVO}"):
    ruta_archivo = f"{RUTA_DCM}/{ARCHIVO}"
elif os.path.exists(f"{RUTA_PNG}/{ARCHIVO}"):
    ruta_archivo = f"{RUTA_PNG}/{ARCHIVO}"
elif os.path.exists(f"{BASE}/{ARCHIVO}"):
    ruta_archivo = f"{BASE}/{ARCHIVO}"
else:
    # Mostrar archivos disponibles si no se encuentra
    print(f"\n   ❌ Archivo no encontrado: {ARCHIVO}")
    print("\n   📂 Archivos disponibles en dicoms/:")
    dcms = sorted([f for f in os.listdir(RUTA_DCM) if f.lower().endswith(".dcm")]) if os.path.exists(RUTA_DCM) else []
    for f in dcms[:15]: print(f"      {f}")
    print("\n   🖼️  Archivos disponibles en pngs/:")
    pngs = sorted([f for f in os.listdir(RUTA_PNG) if f.lower().endswith(".png")]) if os.path.exists(RUTA_PNG) else []
    for f in pngs[:15]: print(f"      {f}")
    raise FileNotFoundError(f"No se encontró: {ARCHIVO}")

ext = ruta_archivo.lower().split(".")[-1]
print(f"   ✅ Archivo encontrado: {ruta_archivo}")
print(f"   📄 Tipo: {ext.upper()}")

# Convertir según tipo
if ext == "dcm":
    print("   🔄 Convirtiendo DICOM → PNG y anonimizando...")
    dcm = pydicom.dcmread(ruta_archivo)
    # Mostrar metadatos antes de anonimizar
    print(f"   📋 Metadatos originales:")
    for tag in ["PatientName","PatientID","PatientBirthDate","StudyDate","Modality"]:
        val = getattr(dcm, tag, "N/A")
        print(f"      {tag}: {val}")
    # Anonimizar
    for tag in ["PatientName","PatientID","PatientBirthDate","PatientSex"]:
        if hasattr(dcm, tag): setattr(dcm, tag, "ANONIMO")
    px = dcm.pixel_array
    if px.ndim == 3: px = px[:,:,0]
    mn,mx = px.min(),px.max()
    if mx>mn: px=((px-mn)/(mx-mn)*255).astype(np.uint8)
    img_orig = Image.fromarray(px,"L")
    print("   ✅ DICOM convertido y anonimizado")
elif ext in ["png","jpg","jpeg"]:
    img_orig = Image.open(ruta_archivo).convert("L")
    print("   ✅ PNG cargado (sin metadatos de paciente)")
else:
    raise ValueError(f"Formato no soportado: {ext}. Usa .dcm, .png o .jpg")

# Preprocesar
arr     = np.array(img_orig.resize((res,res))).astype(np.float32)/255.0
X_pred  = np.expand_dims(np.stack([arr,arr,arr],axis=-1),0)

# Predecir
pred   = model_app.predict(X_pred,verbose=0)[0]
idx_p  = np.argmax(pred)
diag   = cats[idx_p]
conf   = pred[idx_p]*100

print(f"\n   🎯 DIAGNÓSTICO: {diag.upper()}  ({conf:.1f}% confianza)")

# Grad-CAM
def grad_cam(model,img_array,idx_clase):
    try:
        gm = tf.keras.Model(inputs=model.inputs,
             outputs=[model.get_layer("out_relu").output, model.output])
        with tf.GradientTape() as tape:
            co,pr = gm(img_array); score = pr[:,idx_clase]
        w   = tf.reduce_mean(tape.gradient(score,co),axis=(0,1,2))
        cam = tf.reduce_sum(tf.multiply(w,co[0]),axis=-1)
        cam = np.maximum(cam.numpy(),0)
        if cam.max()>0: cam/=cam.max()
        return np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((res,res)))/255.0
    except: return None

cam = grad_cam(model_app,X_pred,idx_p)
n_plots = 3 if cam is not None else 2
fig,axes = plt.subplots(1,n_plots,figsize=(n_plots*4,4))
fig.suptitle(f"DIAGNÓSTICO: {diag.upper()}  —  {conf:.1f}% confianza",
             fontsize=14,fontweight="bold",color="#0D1B40")

axes[0].imshow(arr,cmap="gray")
axes[0].set_title(f"Imagen: {os.path.basename(ARCHIVO)}",fontweight="bold",fontsize=9)
axes[0].axis("off")

colores = ["#C00000" if i==idx_p else "#1A73E8" for i in range(len(cats))]
axes[1].barh(cats,pred*100,color=colores)
axes[1].set_xlabel("Probabilidad (%)")
axes[1].set_title("Probabilidades por clase",fontweight="bold")
for i,p in enumerate(pred): axes[1].text(p*100+0.5,i,f"{p*100:.1f}%",va="center",fontsize=9)

if cam is not None:
    overlay=np.zeros((*arr.shape,4)); overlay[:,:,0]=cam; overlay[:,:,3]=cam*0.6
    axes[2].imshow(arr,cmap="gray"); axes[2].imshow(overlay,cmap="hot",alpha=0.5)
    axes[2].set_title("Grad-CAM\n(zona de decisión IA)",fontweight="bold"); axes[2].axis("off")

plt.tight_layout()
ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
nom_base = os.path.splitext(os.path.basename(ARCHIVO))[0]
ruta_rep = f"{RUTA_RES}/dx_{diag}_{nom_base}_{ts}.png"
fig.savefig(ruta_rep,dpi=150,bbox_inches="tight"); plt.show()

print("\n" + "━"*55)
print(f"  🎯  DIAGNÓSTICO FINAL: {diag.upper()}")
print(f"  Confianza   : {conf:.1f}%")
print()
print("  Todas las probabilidades:")
for c,p in sorted(zip(cats,pred),key=lambda x:-x[1]):
    bar = "█"*int(p*25)
    print(f"    {c:<18} {bar:<25} {p*100:.1f}%")
print()
print(f"  💾 Reporte guardado: {ruta_rep}")
print("  📌 Cambia ARCHIVO arriba y ejecuta de nuevo para otra imagen")
print("━"*55)
